In [ ]:
import os
import random
import shutil
from pathlib import Path
import yaml
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import torch

In [ ]:
# Paths
BASE_DIR = Path("/mnt/e/NCSU/Fall_2025/Board Game/board_keypoint_prediction")
DATASET_DIR = BASE_DIR / "yolo_keypoint_dataset"
IMAGES_DIR = DATASET_DIR / "images_aug"
LABELS_DIR = DATASET_DIR / "labels_aug"
YOLO_DATASET_DIR = BASE_DIR / "yolo_pose_dataset"

In [ ]:
def mask_to_bbox(mask: np.ndarray):
    """
    Converts a binary segmentation mask to rotated bounding box coordinates.
    Returns 4 corner points of the minimum rotated rectangle
    """
    # Ensure binary mask (values 0 or 1)
    mask_binary = (mask > 0.5).astype(np.uint8) * 255
    
    # Find contours
    contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return None
    
    # Get the largest contour
    largest_contour = max(contours, key=cv2.contourArea)
    
    # Get minimum rotated rectangle
    rect = cv2.minAreaRect(largest_contour)
    
    # Get 4 corner points of the rotated rectangle
    box_points = cv2.boxPoints(rect)
    box_points = np.intp(box_points)  # Use np.intp instead of deprecated np.int0
    
    return box_points

In [ ]:
def keypoints_to_yolo_bbox(keypoints_norm, visibilities):
    """
    Calculate bbox directly from visible keypoints.
    
    Args:
        keypoints_norm: List of (x, y) tuples in normalized coords (0-1)
        visibilities: List of visibility flags
    
    Returns:
        (x_center, y_center, width, height) all normalized to 0-1
    """
    # Only use visible keypoints
    visible_kps = np.array([kp for kp, vis in zip(keypoints_norm, visibilities) if vis > 0])
    
    if len(visible_kps) == 0:
        # No visible keypoints - use full image as fallback
        return 0.5, 0.5, 1.0, 1.0
    
    x_min = visible_kps[:, 0].min()
    x_max = visible_kps[:, 0].max()
    y_min = visible_kps[:, 1].min()
    y_max = visible_kps[:, 1].max()
    
    # Calculate center and dimensions
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min
    
    # Add small padding to avoid zero-size boxes
    padding = 0.02
    width = max(width + padding, 0.05)
    height = max(height + padding, 0.05)
    
    # Clamp to valid range
    x_center = np.clip(x_center, 0.0, 1.0)
    y_center = np.clip(y_center, 0.0, 1.0)
    width = np.clip(width, 0.0, 1.0)
    height = np.clip(height, 0.0, 1.0)
    
    return x_center, y_center, width, height

In [ ]:
# def get_board_bbox_from_image(image_path, loaded_model):
#     """
#     Extract board bounding box from image using segmentation model.
    
#     Args:
#         image_path: Path to the image file
#         loaded_model: Loaded YOLO segmentation model
    
#     Returns:
#         tuple: (x_center, y_center, width, height) normalized to 0-1
#         or None if no board detected
#     """
#     from PIL import Image
    
#     # Load image
#     img = Image.open(str(image_path))
#     img_array = np.array(img)
#     h, w = img_array.shape[:2]
    
#     # Run detection
#     results = loaded_model(str(image_path), verbose=False)
#     result = results[0]
    
#     # Extract mask
#     if result.masks is None or len(result.masks) == 0:
#         print(f"⚠️ No board detected in {image_path}")
#         return None
    
#     mask = result.masks[0].data.cpu().numpy()
    
#     # Squeeze to 2D if needed
#     if mask.ndim == 3:
#         mask = mask[0]
    
#     # Resize mask to image dimensions
#     mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
    
#     # Get bbox from mask
#     bblox = mask_to_bbox(mask_resized)
    
#     if bblox is None:
#         print(f"⚠️ Could not extract bbox from mask in {image_path}")
#         return None
    
#     # Convert to YOLO format
#     x_center, y_center, width, height = corners_to_yolo_bbox(bblox, h, w)
    
#     return x_center, y_center, width, height


# # Usage example:
# model_path = "/mnt/e/NCSU/Fall_2025/Board Game/board_segmentation/runs/board_segmentation_80/weights/best.pt"
# loaded_model = YOLO(model_path)

# # Test with a single image
# test_img = random.choice(list(IMAGES_DIR.glob("*.jpg")))
# result = get_board_bbox_from_image(test_img, loaded_model)

In [ ]:
# # convert_to_keypoint.py - CORRECT FORMAT
# import json
# import os
# Image.MAX_IMAGE_PIXELS = None

# # Paths
# input_json = "yolo_keypoint_dataset/yolo_keypoint_dataset.json"
# output_dir = "yolo_keypoint_dataset/labels"
# img_dir = "yolo_keypoint_dataset/images/"
# os.makedirs(output_dir, exist_ok=True)

# model_path = "/mnt/e/NCSU/Fall_2025/Board Game/board_segmentation/runs/board_segmentation_80/weights/best.pt"
# loaded_model = YOLO(model_path)

# # Keypoint configuration
# KEYPOINTS = [f"Hex{i+1}" for i in range(9)]

# with open(input_json) as f:
#     data = json.load(f)

# for item in data:
#     image_name = item["file_upload"].split(".")[0]
#     annos = item["annotations"][0]["result"]

#     kp_dict = {}
#     for kp in annos:
#         if kp["type"] == "keypointlabels":
#             label = kp["value"]["keypointlabels"][0]
#             x = kp["value"]["x"] / 100
#             y = kp["value"]["y"] / 100

#             if "meta" in kp:
#                 if kp["meta"]["text"][0] == "occluded":
#                     viz = 1
#             else:
#                 viz = 2

#             kp_dict[label] = (x, y, viz)
    
#     keypoint_data = []

#     for label in KEYPOINTS:
#         if label in kp_dict:
#             x, y, viz = kp_dict[label]
#             keypoint_data.extend([x, y, viz])
#         else:
#             print("Label not found", label, "in file", image_name)
#             keypoint_data.extend([0.0, 0.0, 0])

#     img_path = img_dir + image_name + '.jpg'
    
#     x_center, y_center, w, h = get_board_bbox_from_image(img_path, loaded_model)

#     txt_path = os.path.join(output_dir, f"{image_name}.txt")
#     with open(txt_path, "w") as f_out:
#         f_out.write(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f} ")

#         for i in range(0, len(keypoint_data), 3):
#             x = keypoint_data[i]
#             y = keypoint_data[i+1]
#             v = keypoint_data[i+2]
#             f_out.write(f"{x:.6f} {y:.6f} {v} ")
        
#         f_out.write("\n")

# print("Conversion completed with CORRECT YOLO pose format!")

# UPDATED CONVERSION CELL - Remove segmentation model dependency
import json
import os
Image.MAX_IMAGE_PIXELS = None

# Paths
input_json = "yolo_keypoint_dataset/yolo_keypoint_dataset.json"
output_dir = "yolo_keypoint_dataset/labels"
img_dir = "yolo_keypoint_dataset/images/"
os.makedirs(output_dir, exist_ok=True)

# Keypoint configuration
KEYPOINTS = [f"Hex{i+1}" for i in range(9)]

with open(input_json) as f:
    data = json.load(f)

for item in data:
    image_name = item["file_upload"].split(".")[0]
    annos = item["annotations"][0]["result"]

    kp_dict = {}
    for kp in annos:
        if kp["type"] == "keypointlabels":
            label = kp["value"]["keypointlabels"][0]
            x = kp["value"]["x"] / 100  # Already normalized (0-1)
            y = kp["value"]["y"] / 100  # Already normalized (0-1)

            # Visibility: 0=not visible, 1=occluded, 2=visible
            viz = 2  # Default to visible
            if "meta" in kp and "text" in kp["meta"]:
                if kp["meta"]["text"][0] == "occluded":
                    viz = 1  # Mark as occluded

            kp_dict[label] = (x, y, viz)
    
    # Extract keypoints in order
    keypoint_data = []
    keypoints_norm = []
    visibilities = []

    for label in KEYPOINTS:
        if label in kp_dict:
            x, y, viz = kp_dict[label]
            keypoint_data.extend([x, y, viz])
            keypoints_norm.append((x, y))
            visibilities.append(viz)
        else:
            print(f"⚠️ Label not found: {label} in file {image_name}")
            keypoint_data.extend([0.0, 0.0, 0])
            keypoints_norm.append((0.0, 0.0))
            visibilities.append(0)

    # ✅ CALCULATE BBOX FROM KEYPOINTS (not segmentation model)
    x_center, y_center, w, h = keypoints_to_yolo_bbox(keypoints_norm, visibilities)

    txt_path = os.path.join(output_dir, f"{image_name}.txt")
    with open(txt_path, "w") as f_out:
        f_out.write(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f} ")

        for i in range(0, len(keypoint_data), 3):
            x = keypoint_data[i]
            y = keypoint_data[i+1]
            v = keypoint_data[i+2]
            f_out.write(f"{x:.6f} {y:.6f} {v} ")
        
        f_out.write("\n")

print("✅ Conversion completed! Bboxes derived from keypoints.")
print(f"✅ Processed {len(data)} images")

In [ ]:
import os
import cv2
import albumentations as A
import numpy as np

# Paths
DATASET_DIR = "yolo_keypoint_dataset"
IMG_DIR = os.path.join(DATASET_DIR, "images")
LBL_DIR = os.path.join(DATASET_DIR, "labels")
OUT_IMG_DIR = os.path.join(DATASET_DIR, "images_aug")
OUT_LBL_DIR = os.path.join(DATASET_DIR, "labels_aug")
os.makedirs(OUT_IMG_DIR, exist_ok=True)
os.makedirs(OUT_LBL_DIR, exist_ok=True)

NUM_KEYPOINTS = 9
N_AUGS = 10

transform = A.Compose([
    A.Perspective(scale=(0.05, 0.1), keep_size=True, pad_mode=cv2.BORDER_CONSTANT, p=0.3),
    A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.7),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussianBlur(blur_limit=3, p=0.2)
], keypoint_params=A.KeypointParams(format='xy', remove_invisible=False))

def read_yolo_keypoint_label(label_path):
    with open(label_path, "r") as f:
        line = f.readline().strip().split()
        class_id = int(line[0])
        bbox = [float(x) for x in line[1:5]]

        keypoints = []
        visibilities = []
        for i in range(5, len(line), 3):
            if i+2 < len(line):
                x = float(line[i])
                y = float(line[i+1])
                v = int(line[i+2])
                keypoints.append((x, y))
                visibilities.append(v)
        
        return class_id, bbox, keypoints, visibilities

def write_yolo_keypoint_label(label_path, class_id, bbox, keypoints, visibilities):
    with open(label_path, "w") as f:
        f.write(f"{class_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f} ")

        for i in range(len(keypoints)):
            x, y = keypoints[i]
            v = visibilities[i]
            f.write(f"{x:.6f} {y:.6f} {v} ")
        
        f.write("\n")

def recalculate_bbox(keypoints, visibilities):
    """
    Recalculate bbox from keypoints.
    Only considers visible keypoints (v > 0).
    """
    visible_kps = [kp for kp, vis in zip(keypoints, visibilities) if vis > 0]
    
    if not visible_kps:
        # No visible keypoints, return default centered bbox
        return 0.5, 0.5, 1.0, 1.0
    
    visible_kps = np.array(visible_kps)
    x_min = visible_kps[:, 0].min()
    x_max = visible_kps[:, 0].max()
    y_min = visible_kps[:, 1].min()
    y_max = visible_kps[:, 1].max()
    
    # Add padding around keypoints
    padding = 0.05
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = max((x_max - x_min) + padding, 0.1)
    height = max((y_max - y_min) + padding, 0.1)
    
    # Clamp to valid range
    x_center = max(0.0, min(1.0, x_center))
    y_center = max(0.0, min(1.0, y_center))
    width = min(1.0, width)
    height = min(1.0, height)
    
    return x_center, y_center, width, height

img_files = [f for f in os.listdir(IMG_DIR) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

for img_file in img_files:
    img_path = os.path.join(IMG_DIR, img_file)
    lbl_path = os.path.join(LBL_DIR, os.path.splitext(img_file)[0] + ".txt")
    if not os.path.exists(lbl_path):
        continue

    image = cv2.imread(img_path)
    h, w = image.shape[:2]
    class_id, bbox, keypoints, visibilities = read_yolo_keypoint_label(lbl_path)

    pixel_keypoints = []
    for idx, (kp, vis) in enumerate(zip(keypoints, visibilities)):
        x, y = kp
        if vis in [1, 2]:
            pixel_keypoints.append((x * w, y * h))
        else:
            pixel_keypoints.append((0.0, 0.0))

    for i in range(N_AUGS):
        transformed = transform(image=image, keypoints=pixel_keypoints)
        aug_img = transformed["image"]
        aug_kps = transformed["keypoints"]

        aug_keypoints = []
        aug_vis = []
        
        for idx, (aug_kp, original_vis) in enumerate(zip(aug_kps, visibilities)):
            x, y = aug_kp
            
            # if original_vis in [1, 2]:
            #     norm_x = max(0.0, min(1.0, x / aug_img.shape[1]))
            #     norm_y = max(0.0, min(1.0, y / aug_img.shape[0]))
            #     aug_keypoints.append((norm_x, norm_y))
            #     aug_vis.append(original_vis)
            # else:
            #     aug_keypoints.append((0.0, 0.0))
            #     aug_vis.append(0)

            h_aug, w_aug = aug_img.shape[:2]
            # Check if keypoint is within image bounds
            if 0 <= x < w_aug and 0 <= y < h_aug and original_vis in [1, 2]:
                norm_x = x / w_aug
                norm_y = y / h_aug
                aug_keypoints.append((norm_x, norm_y))
                aug_vis.append(original_vis)
            else:
                # Keypoint is outside bounds or not visible - mark as fake
                aug_keypoints.append((0.0, 0.0))
                aug_vis.append(0)

        # ✅ RECALCULATE BBOX AFTER AUGMENTATION
        aug_bbox = recalculate_bbox(aug_keypoints, aug_vis)

        out_img_file = os.path.splitext(img_file)[0] + f"_aug{i}.jpg"
        out_lbl_file = os.path.splitext(img_file)[0] + f"_aug{i}.txt"
        cv2.imwrite(os.path.join(OUT_IMG_DIR, out_img_file), aug_img)
        write_yolo_keypoint_label(
            os.path.join(OUT_LBL_DIR, out_lbl_file),
            class_id, aug_bbox, aug_keypoints, aug_vis
        )

print("Data augmentation completed successfully!")
print("✅ Keypoints and bboxes updated after each augmentation")

In [ ]:
KEYPOINTS = 9

KEYPOINT_NAMES = [f"Hex{i+1}" for i in range(KEYPOINTS)]
CLASS_NAMES = ["board"]

In [ ]:
image_files = sorted([f for f in IMAGES_DIR.glob("*.jpg")])
image_names = [img.stem for img in image_files]

random.seed(42)
random.shuffle(image_names)

n_total = len(image_names)
n_train = int(0.7 * n_total)
n_val = int(0.2 * n_total)
n_test = n_total - n_train - n_val

train_names = image_names[:n_train]
val_names = image_names[n_train:n_train+n_val]
test_names = image_names[n_train+n_val:]

In [ ]:
def create_yolo_structure():
    if YOLO_DATASET_DIR.exists():
        shutil.rmtree(YOLO_DATASET_DIR)
    for split in ['train', 'val', 'test']:
        (YOLO_DATASET_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
        (YOLO_DATASET_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)
    print("Created YOLO pose dataset directory structure.")

create_yolo_structure()

In [ ]:
def copy_files_to_splits():
    splits = {
        'train': train_names,
        'val': val_names,
        'test': test_names
    }
    for split_name, file_names in splits.items():
        for file_name in file_names:
            src_img = IMAGES_DIR / f"{file_name}.jpg"
            dst_img = YOLO_DATASET_DIR / split_name / 'images' / f"{file_name}.jpg"
            shutil.copy2(src_img, dst_img)
            src_label = LABELS_DIR / f"{file_name}.txt"
            dst_label = YOLO_DATASET_DIR / split_name / 'labels' / f"{file_name}.txt"
            shutil.copy2(src_label, dst_label)
    print("Copied images and labels to splits.")

copy_files_to_splits()

In [ ]:
def create_yolo_pose_config():
    config = {
        'path': str(YOLO_DATASET_DIR.resolve()),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'names': CLASS_NAMES,
        'kpt_shape': [KEYPOINTS, 3],
        'keypoints': KEYPOINT_NAMES
    }
    config_file = YOLO_DATASET_DIR / 'dataset.yaml'
    with open(config_file, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    print(f"Created YOLO pose configuration file: {config_file}")
    return config_file

config_file = create_yolo_pose_config()

In [ ]:
def plot_pose_sample(img_path, label_path, ax, keypoint_names=KEYPOINT_NAMES):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    
    with open(label_path, 'r') as f:
        line = f.readline().strip().split()

        kpts = []
        vis = []
        for i in range(5, len(line), 3):
            if i+2 < len(line):
                kpts.extend([float(line[i]), float(line[i+1])])
                vis.append(int(line[i+2]))
    
    h, w = img.shape[:2]
    for i in range(KEYPOINTS):
        x = kpts[2*i] * w
        y = kpts[2*i+1] * h
        v = vis[i]
        if v > 0:
            ax.scatter(x, y, s=60, c='red', marker='o')
            ax.text(x, y, keypoint_names[i], color='blue', fontsize=10)
    ax.set_title(f"{img_path.name}")
    ax.axis('off')

In [ ]:
sample_names = random.sample(train_names, min(4, len(train_names)))
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, sample_name in enumerate(sample_names):
    img_path = YOLO_DATASET_DIR / 'train' / 'images' / f"{sample_name}.jpg"
    label_path = YOLO_DATASET_DIR / 'train' / 'labels' / f"{sample_name}.txt"
    plot_pose_sample(img_path, label_path, axes[i])
plt.tight_layout()
plt.show()
print("Sample visualization completed. Red dots show keypoints.")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO('yolov8m-pose.pt')

EPOCHS = 80
IMAGE_SIZE = 416
BATCH_SIZE = 8

results = model.train(
    data=str(config_file),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=device,
    name="board_pose",
    exist_ok=True,
    verbose=True,
    workers=2,
    single_cls=True,
)

print(f"Model saved to: {results.save_dir}")

In [ ]:
best_model_path = Path("runs/pose/board_pose/weights/best.pt")
pose_model = YOLO(str(best_model_path))

sample_names = random.sample(test_names, min(4, len(test_names)))
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, sample_name in enumerate(sample_names):
    img_path = YOLO_DATASET_DIR / 'test' / 'images' / f"{sample_name}.jpg"
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    axes[i].set_title(f"Predicted: {img_path.name}")
    axes[i].axis('off')

    results = pose_model(str(img_path), verbose=False)
    result = results[0]
    print(result.keypoints)
    h, w = img.shape[:2]

    if hasattr(result, "keypoints") and result.keypoints is not None:
        kpts = result.keypoints.xy[0]
        confs = result.keypoints.conf[0]
        for j, (xy, conf) in enumerate(zip(kpts, confs)):
            x, y = xy.cpu().numpy()
            if conf > 0.9:
                axes[i].scatter(x, y, s=60, c='lime', marker='o')
                axes[i].text(x, y, KEYPOINT_NAMES[j], color='blue', fontsize=10)
    else:
        axes[i].text(0.5, 0.5, "No keypoints detected", ha='center', va='center', transform=axes[i].transAxes, color='red')

plt.tight_layout()
plt.show()
print("Inference and visualization completed. Line dots show predicted keypoints.")

Keypoint Prediction Metrics & Evaluation

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import json

def read_label_file(label_path):
    """Read YOLO pose label file and extract keypoints."""
    with open(label_path, 'r') as f:
        line = f.readline().strip().split()
        class_id = int(line[0])
        bbox = [float(x) for x in line[1:5]]
        
        keypoints = []
        visibilities = []
        for i in range(5, len(line), 3):
            if i+2 < len(line):
                x = float(line[i])
                y = float(line[i+1])
                v = int(line[i+2])
                keypoints.append((x, y))
                visibilities.append(v)
        
        return np.array(keypoints), np.array(visibilities), bbox

def evaluate_keypoint_predictions(test_names, model, dataset_dir, keypoint_names):
    """
    Evaluate model predictions against ground truth.
    
    Returns:
        dict: Metrics for each keypoint and overall
    """
    metrics = {
        'per_keypoint': {},
        'overall': {}
    }
    
    all_predictions = []
    all_ground_truth = []
    all_distances = []
    
    for sample_name in test_names:
        img_path = dataset_dir / 'test' / 'images' / f"{sample_name}.jpg"
        label_path = dataset_dir / 'test' / 'labels' / f"{sample_name}.txt"
        
        if not label_path.exists():
            continue
        
        # Read ground truth
        gt_keypoints, gt_vis, _ = read_label_file(label_path)
        
        # Get predictions
        results = model(str(img_path), verbose=False)
        result = results[0]
        
        if not hasattr(result, "keypoints") or result.keypoints is None:
            continue
        
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        
        # Extract predicted keypoints (normalized)
        pred_kpts = result.keypoints.xyn[0].cpu().numpy()  # Use normalized coords
        pred_confs = result.keypoints.conf[0].cpu().numpy()
        
        # Calculate per-keypoint metrics
        for j in range(len(gt_keypoints)):
            if gt_vis[j] > 0:  # Only evaluate visible keypoints
                gt_x, gt_y = gt_keypoints[j]
                pred_x, pred_y = pred_kpts[j]
                conf = pred_confs[j]
                
                # Euclidean distance in normalized space
                distance = np.sqrt((gt_x - pred_x)**2 + (gt_y - pred_y)**2)
                all_distances.append(distance)
                
                kp_name = keypoint_names[j]
                if kp_name not in metrics['per_keypoint']:
                    metrics['per_keypoint'][kp_name] = {
                        'distances': [],
                        'confidences': [],
                        'count': 0
                    }
                
                metrics['per_keypoint'][kp_name]['distances'].append(distance)
                metrics['per_keypoint'][kp_name]['confidences'].append(float(conf))
                metrics['per_keypoint'][kp_name]['count'] += 1
                
                all_predictions.append([pred_x, pred_y])
                all_ground_truth.append([gt_x, gt_y])
    
    # Calculate per-keypoint statistics
    for kp_name, data in metrics['per_keypoint'].items():
        distances = data['distances']
        confs = data['confidences']
        
        metrics['per_keypoint'][kp_name].update({
            'mae': np.mean(distances),
            'rmse': np.sqrt(np.mean(np.array(distances)**2)),
            'std': np.std(distances),
            'min': np.min(distances),
            'max': np.max(distances),
            'avg_confidence': np.mean(confs)
        })
    
    # Calculate overall metrics
    if all_distances:
        all_distances = np.array(all_distances)
        all_predictions = np.array(all_predictions)
        all_ground_truth = np.array(all_ground_truth)
        
        metrics['overall'] = {
            'mae': np.mean(all_distances),
            'rmse': np.sqrt(np.mean(all_distances**2)),
            'std': np.std(all_distances),
            'median': np.median(all_distances),
            'percentile_90': np.percentile(all_distances, 90),
            'percentile_95': np.percentile(all_distances, 95),
            'total_keypoints': len(all_distances),
            'success_rate': np.sum(all_distances < 0.1) / len(all_distances)  # Within 10% of image
        }
    
    return metrics

# Run evaluation
evaluation_metrics = evaluate_keypoint_predictions(
    test_names, 
    pose_model, 
    YOLO_DATASET_DIR,
    KEYPOINT_NAMES
)

# Print results
print("\n" + "="*60)
print("KEYPOINT PREDICTION EVALUATION RESULTS")
print("="*60)

print("\n📊 OVERALL METRICS:")
for metric, value in evaluation_metrics['overall'].items():
    if isinstance(value, float):
        print(f"  {metric:.<30} {value:.6f}")
    else:
        print(f"  {metric:.<30} {value}")

print("\n📍 PER-KEYPOINT METRICS:")
for kp_name, data in evaluation_metrics['per_keypoint'].items():
    print(f"\n  {kp_name}:")
    print(f"    MAE (pixels):        {data['mae']:.6f}")
    print(f"    RMSE (pixels):       {data['rmse']:.6f}")
    print(f"    Std Dev:             {data['std']:.6f}")
    print(f"    Range:               [{data['min']:.6f}, {data['max']:.6f}]")
    print(f"    Avg Confidence:      {data['avg_confidence']:.4f}")
    print(f"    Samples:             {data['count']}")

In [ ]:
def visualize_predictions_vs_ground_truth(test_names, model, dataset_dir, keypoint_names, num_samples=4):
    """Visualize predicted vs ground truth keypoints side-by-side."""
    sample_names = random.sample([s for s in test_names if (dataset_dir / 'test' / 'labels' / f"{s}.txt").exists()], 
                                 min(num_samples, len(test_names)))
    
    fig, axes = plt.subplots(num_samples, 2, figsize=(14, 5*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for idx, sample_name in enumerate(sample_names):
        img_path = dataset_dir / 'test' / 'images' / f"{sample_name}.jpg"
        label_path = dataset_dir / 'test' / 'labels' / f"{sample_name}.txt"
        
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Ground truth
        gt_keypoints, gt_vis, _ = read_label_file(label_path)
        axes[idx, 0].imshow(img_rgb)
        axes[idx, 0].set_title(f"Ground Truth: {sample_name}")
        axes[idx, 0].axis('off')
        
        for j, (kp, vis) in enumerate(zip(gt_keypoints, gt_vis)):
            if vis > 0:
                x, y = int(kp[0] * w), int(kp[1] * h)
                axes[idx, 0].scatter(x, y, s=80, c='red', marker='o', edgecolors='white', linewidth=2)
                axes[idx, 0].text(x+5, y+5, keypoint_names[j], color='red', fontsize=9, weight='bold')
        
        # Predictions
        results = pose_model(str(img_path), verbose=False)
        result = results[0]
        axes[idx, 1].imshow(img_rgb)
        axes[idx, 1].set_title(f"Predictions: {sample_name}")
        axes[idx, 1].axis('off')
        
        if hasattr(result, "keypoints") and result.keypoints is not None:
            pred_kpts = result.keypoints.xyn[0].cpu().numpy()
            pred_confs = result.keypoints.conf[0].cpu().numpy()
            
            for j, (kp, conf) in enumerate(zip(pred_kpts, pred_confs)):
                x, y = int(kp[0] * w), int(kp[1] * h)
                color = 'lime' if conf > 0.8 else 'yellow'
                axes[idx, 1].scatter(x, y, s=80, c=color, marker='o', edgecolors='white', linewidth=2)
                axes[idx, 1].text(x+5, y+5, f"{keypoint_names[j]} ({conf:.2f})", 
                                 color=color, fontsize=9, weight='bold')
    
    plt.tight_layout()
    plt.show()

# Visualize predictions
visualize_predictions_vs_ground_truth(test_names, pose_model, YOLO_DATASET_DIR, KEYPOINT_NAMES, num_samples=4)